### Import packages and data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import re
import spacy
from nltk.corpus import wordnet as wn

/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [ ]:
# Load English spacy model
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

/Users/tildeidunsloth/Desktop/Thesis/Thesis_project/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PROJECT_ROOT = Path("/Users/tildeidunsloth/Desktop/Thesis")
SRL_DIR = PROJECT_ROOT / "data" / "SRL"
GENDERED_NAMES_DIR = PROJECT_ROOT / "data" / "gendered_names"

In [4]:
romance_srl = pd.read_csv(SRL_DIR / "romance_srl.csv")
lit_fic_srl = pd.read_csv(SRL_DIR / "literary_fiction_srl.csv")
sci_fi_srl = pd.read_csv(SRL_DIR / "sci_fi_srl.csv")

In [5]:
# Load previously extracted gendered names
with open(GENDERED_NAMES_DIR / "male_names_sci_fi.json") as f:
    male_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_sci_fi.json") as f:
    female_names_sci_fi = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_lit.json") as f:
    male_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "female_names_lit.json") as f:
    female_names_lit = json.load(f)

with open(GENDERED_NAMES_DIR / "male_names_romance.json") as f:
    male_names_romance = json.load(f)
    
with open(GENDERED_NAMES_DIR / "female_names_romance.json") as f:
    female_names_romance = json.load(f)

### Gender information

In [6]:
female_list = ['she', 'daughter', 'hers', 'her', 'mother', 'woman', 'girl', 'herself', 'female', 'sister', 'daughters', 'mothers', 'women', 'girls', 'females', 'sisters', 'aunt', 'aunts', 'niece', 'nieces', 'mrs.']
male_list = ['he', 'son', 'his', 'him', 'father', 'man', 'boy', 'himself', 'male', 'brother', 'sons', 'fathers', 'men', 'boys', 'males', 'brothers', 'uncle', 'uncles', 'nephew', 'nephews', 'mr.']

In [7]:
romance_female = female_list + female_names_romance
romance_male = male_list + male_names_romance
romance_names = romance_female + romance_male

lit_fic_female = female_list + female_names_lit
lit_fic_male = male_list + male_names_lit
lit_fic_names = lit_fic_female + lit_fic_male

sci_fi_female = female_list + female_names_sci_fi
sci_fi_male = male_list + male_names_sci_fi
sci_fi_names = sci_fi_female + sci_fi_male

In [ ]:
def classify_gender(text, female_set, male_set):
    if pd.isna(text):
        return None
    
    tokens = re.findall(r"\b\w+\b", text.lower())
    
    has_female = any(t in female_set for t in tokens)
    has_male = any(t in male_set for t in tokens)
    
    if has_female and not has_male:
        return "female"
    elif has_male and not has_female:
        return "male"
    elif has_male and has_female:
        return "mixed"
    else:
        return None

In [9]:
romance_female_set = set(w.lower() for w in romance_female)
romance_male_set = set(w.lower() for w in romance_male)
romance_set = set(w.lower() for w in romance_names)

lit_fic_female_set = set(w.lower() for w in lit_fic_female)
lit_fic_male_set = set(w.lower() for w in lit_fic_male)
lit_fic_set = set(w.lower() for w in lit_fic_names)

sci_fi_female_set = set(w.lower() for w in sci_fi_female)
sci_fi_male_set = set(w.lower() for w in sci_fi_male)
sci_fi_set = set(w.lower() for w in sci_fi_names)

In [10]:
for col in ["agent", "patient", "receiver"]:
    romance_srl[col + "_gender"] = romance_srl[col].apply(
        lambda x: classify_gender(x, romance_female_set, romance_male_set)
    )

for col in ["agent", "patient", "receiver"]:
    lit_fic_srl[col + "_gender"] = lit_fic_srl[col].apply(
        lambda x: classify_gender(x, lit_fic_female_set, lit_fic_male_set)
    )

for col in ["agent", "patient", "receiver"]:
    sci_fi_srl[col + "_gender"] = sci_fi_srl[col].apply(
        lambda x: classify_gender(x, sci_fi_female_set, sci_fi_male_set)
    )

In [11]:
cols = ["agent_gender", "patient_gender", "receiver_gender"]

df_no_gender_romance = romance_srl[romance_srl[cols].isna().all(axis=1)]

df_no_gender_lit_fic = lit_fic_srl[lit_fic_srl[cols].isna().all(axis=1)]

df_no_gender_sci_fi = sci_fi_srl[sci_fi_srl[cols].isna().all(axis=1)]

### Cleaning

In [12]:
# remove rows with no gender information
romance_srl = romance_srl[~romance_srl.index.isin(df_no_gender_romance.index)]

lit_fic_srl = lit_fic_srl[~lit_fic_srl.index.isin(df_no_gender_lit_fic.index)]

sci_fi_srl = sci_fi_srl[~sci_fi_srl.index.isin(df_no_gender_sci_fi.index)]

In [13]:
# remove names from the predicate column
def remove_names_from_predicate(df, name_set):
    def remove_names(text):
        if pd.isna(text):
            return text
        
        tokens = re.findall(r"\b\w+\b", text)
        filtered_tokens = [t for t in tokens if t.lower() not in name_set]
        return " ".join(filtered_tokens)
    
    df["predicate"] = df["predicate"].apply(remove_names)
    return df

In [14]:
romance_srl = remove_names_from_predicate(romance_srl, romance_set)
lit_fic_srl = remove_names_from_predicate(lit_fic_srl, lit_fic_set)
sci_fi_srl = remove_names_from_predicate(sci_fi_srl, sci_fi_set)

In [15]:
# replace predicate "roze" with "froze"
romance_srl["predicate"] = romance_srl["predicate"].replace({"roze": "froze"})
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].replace({"roze": "froze"})
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].replace({"roze": "froze"})

In [16]:
# replace predicate "ase" with "ease"
romance_srl["predicate"] = romance_srl["predicate"].replace({"ased": "eased"})
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].replace({"ased": "eased"})
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].replace({"ased": "eased"})

romance_srl["predicate"] = romance_srl["predicate"].replace({"ase": "ease"})
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].replace({"ase": "ease"})
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].replace({"ase": "ease"})

In [17]:
# replace predicate "humbed" with "thumbed"
romance_srl["predicate"] = romance_srl["predicate"].replace({"humbed": "thumbed"})
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].replace({"humbed": "thumbed"})
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].replace({"humbed": "thumbed"})

In [18]:
# remove "ve" from predicates like "ve been"
romance_srl["predicate"] = romance_srl["predicate"].str.replace(r"\bve\b", "", regex=True)
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].str.replace(r"\bve\b", "", regex=True)
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].str.replace(r"\bve\b", "", regex=True)

In [19]:
# remove "Sera" from predicates
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].str.replace(r"\bSera\b", "", regex=True)

In [20]:
# remove spaces before the first letter in the predicate
romance_srl["predicate"] = romance_srl["predicate"].str.lstrip()
lit_fic_srl["predicate"] = lit_fic_srl["predicate"].str.lstrip()
sci_fi_srl["predicate"] = sci_fi_srl["predicate"].str.lstrip()

### Add lexname information

In [21]:
def get_lexname(lemma):
    if not isinstance(lemma, str) or lemma.strip() == "":
        return None
    
    synsets = wn.synsets(lemma, pos=wn.VERB)
    
    if not synsets:
        return None
    
    return synsets[0].lexname()

In [22]:
predicates_romance = romance_srl["predicate"].fillna("").astype(str).tolist()

docs_romance = list(nlp.pipe(predicates_romance, batch_size=2000))

romance_srl["predicate_lemma"] = [
    doc[0].lemma_ if len(doc) > 0 else None
    for doc in docs_romance
]

In [23]:
lemmas_romance = romance_srl["predicate_lemma"].dropna().unique()

lemma_to_lexname_romance = {}

for lemma in lemmas_romance:
    lemma_to_lexname_romance[lemma] = get_lexname(lemma)

romance_srl["verb_lexname"] = romance_srl["predicate_lemma"].map(lemma_to_lexname_romance)

In [24]:
predicates_lit_fic = lit_fic_srl["predicate"].fillna("").astype(str).tolist()

docs_lit_fic = list(nlp.pipe(predicates_lit_fic, batch_size=2000))

lit_fic_srl["predicate_lemma"] = [
    doc[0].lemma_ if len(doc) > 0 else None
    for doc in docs_lit_fic
]

In [25]:
lemmas_lit_fic = lit_fic_srl["predicate_lemma"].dropna().unique()

lemma_to_lexname_lit_fic = {}

for lemma in lemmas_lit_fic:
    lemma_to_lexname_lit_fic[lemma] = get_lexname(lemma)

lit_fic_srl["verb_lexname"] = lit_fic_srl["predicate_lemma"].map(lemma_to_lexname_lit_fic)

In [26]:
predicates_sci_fi = sci_fi_srl["predicate"].fillna("").astype(str).tolist()

docs_sci_fi = list(nlp.pipe(predicates_sci_fi, batch_size=2000))

sci_fi_srl["predicate_lemma"] = [
    doc[0].lemma_ if len(doc) > 0 else None
    for doc in docs_sci_fi
    ]

In [27]:
lemmas_sci_fi = sci_fi_srl["predicate_lemma"].dropna().unique()

lemma_to_lexname_sci_fi = {}

for lemma in lemmas_sci_fi:
    lemma_to_lexname_sci_fi[lemma] = get_lexname(lemma)

sci_fi_srl["verb_lexname"] = sci_fi_srl["predicate_lemma"].map(lemma_to_lexname_sci_fi)

In [28]:
# save csv to SRL_DIR
romance_srl.to_csv(SRL_DIR / "romance_srl_cleaned.csv", index=False)
lit_fic_srl.to_csv(SRL_DIR / "literary_fiction_srl_cleaned.csv", index=False)
sci_fi_srl.to_csv(SRL_DIR / "sci_fi_srl_cleaned.csv", index=False)